# Problem 02: 
Chuẩn hoá dữ liệu về phân phối có kỳ vọng là 0 và phương sai là 1.
So sánh kết quả của mô hình sau khi được huấn luyện với dữ liệu đã được chuẩn hoá so với kết quả trong bài 1

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Import dataset
df = pd.read_csv('forestfires.csv')
df.head(10)

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0
7,8,6,aug,mon,91.5,145.4,608.2,10.7,8.0,86,2.2,0.0,0.0
8,8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0,0.0
9,7,5,sep,sat,92.5,88.0,698.6,7.1,22.8,40,4.0,0.0,0.0


In [4]:
#  Covert days into numerical values

def convert_day(day: str)->int:
    day_dict = {
        'mon' : 1,
        'tue' : 2,
        'wed' : 3,
        'thu' : 4,
        'fri' : 5,
        'sat' : 6,
        'sun' : 7,
    }
    return day_dict[day]

In [5]:
# Covert months into numerical values

def convert_month(month: str)->int:
    month_dict = {
        "jan" : 1,
        "feb" : 2,
        "mar" : 3,
        "apr" : 4,
        "may" : 5,
        "jun" : 6,
        "jul" : 7,
        "aug" : 8,
        "sep" : 9,
        "oct" : 10,
        "nov" : 11,
        "dec" : 12,
    }
    return month_dict[month]

In [6]:
df['day'] = df['day'].apply(convert_day)
df['month'] = df['month'].apply(convert_month)
df.head(5)

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,3,5,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,10,2,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,10,6,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,3,5,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,3,7,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


In [ ]:
# Standardize values in FFM, DC, DMC  column
df['FFMC'] = (df['FFMC'] - df['FFMC'].mean()) / df['FFMC'].std()
df['DC'] = (df['DC'] - df['DC'].mean()) / df['DC'].std()
df['DMC'] = (df['DMC'] - df['DMC'].mean()) / df['DMC'].std()

In [8]:
# Standardize using sklearn
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
columns_to_standardize = ['ISI', 'temp', 'RH', 'wind', 'rain']
df[columns_to_standardize] = scaler.fit_transform(df[columns_to_standardize])

In [9]:
# Using log transformation to normalize because 'area' (skewed - lean toward 0.0) is not following normal distribution
df['area'] = np.log1p(df['area'])

In [10]:
X_y = df.to_numpy() # Cover DataFrame to numpy array

In [11]:
N = df.shape[0]
# Divide data into 80% traing data and 20% testing data
X_y_train, X_y_test = np.split(X_y, indices_or_sections=[int(0.8*N)] )

In [12]:
X_train, X_test = X_y_train[:, :-1], X_y_test[:, :-1]
Y_train, Y_test = X_y_train[:, -1], X_y_test[:, -1]


In [16]:
print(X_train.shape)  
print(X_test.shape)   
print(Y_train.shape)
print(Y_test.shape)

(413, 12)
(104, 12)
(413,)
(104,)


In [13]:
# Algorithm
class LinearRegression:
    def __init__(self):
        self.theta = None
        
    def rmse(self, y: np.ndarray, y_hat:np.ndarray) -> float: # Loss function
        # Root Mean Squared Error 
        delta = y - y_hat
        return np.square(delta).mean()**0.5
    
    def fit(self, X: np.ndarray, Y: np.ndarray) -> None: # Training model
        cov = np.matmul(X.T, X)
        inv_cov = np.linalg.inv(cov)
        self.theta = np.matmul(np.matmul(inv_cov, X.T), Y)
        
    def predict(self, X: np.ndarray) -> np.ndarray:
        y_pred = np.matmul(self.theta.T, X.T)
        return y_pred
    def evaluate(self, X: np.ndarray, Y: np.ndarray) -> float:
        y_pred = self.predict(X)
        return self.rmse(Y, y_pred)

In [14]:
model = LinearRegression()
# Training model
model.fit(X_train, Y_train)

# Predict in testing dataset
y_pred = model.predict(X_test)

# Evaluate model
rmse_value = model.evaluate(X_test, Y_test)
print(f'RMSE: {rmse_value:.6f}')


RMSE: 1.912979


# Nhận xét: RMSE của dữ liệu đã chuẩn hoá thấp hơn nhiều so với dữ liệu gốc -> Việc chuẩn hoá giúp cải thiện mô hình hơn
